# CompSet Studio — Aketa Colab → Drive canary

## Goal
Run one finite public calendar collection on standard CPU and save sanitized JSON/CSV plus digest evidence in **ankitg.owa** Drive. Not executed yet.

Current adapter: Aketa, INR, one adult, zero children, requested one room, 30 dates. Prices are indicative minima, not final payable quotes.

Requested follow-on scope remains **Aketa + six evidenced peers**, 365-day minimums, 30-day OTA details and ten monthly breakup checks. This canary does not implement or claim those uncollected observations.

## Setup
### 1. Verify storage and explicitly mount Drive
Use the named account and standard CPU. Google requires normal sign-in/consent. No paid upgrade, remote runtime, proxy or keepalive. Drive consent allows notebook code broad file access; this reviewed package only writes its new run folder.

In [ ]:
from google.colab import auth, drive
from googleapiclient.discovery import build
import google.auth
from pathlib import Path
import json

auth.authenticate_user()
credentials, _ = google.auth.default()
about = build('drive', 'v3', credentials=credentials).about().get(
    fields='user(emailAddress),storageQuota').execute()
assert about['user']['emailAddress'].lower() == 'ankitg.owa@gmail.com', 'Wrong Drive account: stop here'
quota = about.get('storageQuota', {})
limit = int(quota.get('limit', 0))
assert limit >= 5_000_000_000_000, '5 TB entitlement not verified: stop here'
print({'account': about['user']['emailAddress'], 'quota_bytes': limit,
       'used_bytes': quota.get('usage'), 'runtime': 'standard CPU only'})
# Google asks for explicit consent; this permits this notebook to access Drive.
# The reviewed code below only writes into its new CompSetStudio/collection-runs folder.
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/CompSetStudio/collection-runs')
assert Path('/content/drive/MyDrive').is_dir(), 'Drive mount missing'


### 2. Install pinned public dependencies

In [ ]:
import subprocess, sys
assert 'google.colab' in sys.modules, 'Run this notebook in Colab, not the laptop'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                'scrapling[fetchers]==0.4.15', 'playwright==1.63.0'],
               check=True, timeout=600)
subprocess.run([sys.executable, '-m', 'playwright', 'install', '--with-deps', 'chromium'],
               check=True, timeout=600)
print('Pinned collector dependencies installed; no model or paid API')


### 3. Restore the reviewed package
Only the Windows browser executable is replaced by Playwright-managed Chromium. The original parser and source-stop rules stay intact. Original and derived hashes make that portability change auditable. No credentials, raw conversations or private pairing material are embedded.

In [ ]:
import hashlib, json, sys
from pathlib import Path
PACKAGE_ROOT = Path("/content/compset-colab-package")
source_manifest = {'hotel_aketa.py': {'original_sha256': 'afa3223071c6276d106f2d8dc046995d92d0cbf148448aaae50536808a938f3c', 'packaged_sha256': 'afa3223071c6276d106f2d8dc046995d92d0cbf148448aaae50536808a938f3c'}, 'google_calendar_network.py': {'original_sha256': '212b0ada4c6588866d6d2614d0a9e6bfbbc6a86e090683972c43c2d480800607', 'packaged_sha256': '3a1d0d350fc8e98ac61a03c58299d789cf66009b08f9c1c0083b44944dd7af3b'}}
modules = {'compset/__init__.py': '', 'compset/collect.py': 'from contextlib import contextmanager\nimport logging\n@contextmanager\ndef _quiet_scrapling():\n    previous = logging.root.manager.disable\n    logging.disable(logging.CRITICAL)\n    try:\n        yield\n    finally:\n        logging.disable(previous)\n', 'compset/hotel_aketa.py': '"""Hotel Aketa public Google Hotels calendar evidence.\n\nCalendar minimums and Google partner offers are indicative displays, not direct\nsupplier checkout quotes. Missing prices are always unknown. No credentials,\nopaque RPC requests, outbound booking links, or personal reviews are exported.\n"""\nfrom __future__ import annotations\n\nimport csv\nfrom datetime import date, datetime, timedelta, timezone\nfrom decimal import Decimal, InvalidOperation\nimport json\nfrom pathlib import Path\nimport re\n\n# India currently uses UTC+05:30 without daylight saving; this collector only\n# requests present/future dates. A fixed offset avoids missing Windows tzdata.\nINDIA = timezone(timedelta(hours=5, minutes=30), \'Asia/Kolkata\')\nHOTEL_NAME = \'Hotel Aketa\'\nENTITY = \'ChgI98PmmYGh5fJgGgwvZy8xMmNueDRyN3IQAQ\'\nSOURCE_URL = \'https://www.google.com/travel/hotels/entity/\' + ENTITY\nROOT = Path(__file__).resolve().parents[1]\nDEFAULT_OUTPUT = ROOT / \'data\' / \'hotels\' / \'aketa\'\nMONTHS = \'January February March April May June July August September October November December\'.split()\nMONTH_PATTERN = re.compile(r\'^(\' + \'|\'.join(MONTHS) + r\')(?: (20\\d\\d))?$\')\nPRICE = re.compile(r\'^₹\\s*([0-9]+(?:,[0-9]{3})*(?:\\.[0-9]+)?)(K)?$\')\nTRAVELERS = \'Number of travelers. Current number of travelers is 1.\'\nPARTNERS = {\'Booking.com\', \'Agoda\', \'MakeMyTrip.com\', \'EaseMyTrip.com\', HOTEL_NAME,\n            \'Expedia\', \'Hotels.com\', \'Goibibo.com\', \'Cleartrip.com\', \'Trip.com\'}\n\n\ndef requested_context(start_date=None, days=30):\n    if type(days) is not int or days != 30:\n        raise ValueError(\'Hotel evidence requires exactly 30 consecutive dates\')\n    start = date.fromisoformat(start_date) if start_date else datetime.now(INDIA).date()\n    return {\'hotel_name\': HOTEL_NAME, \'provider_hotel_id\': ENTITY, \'source\': \'google_hotels\',\n            \'timezone\': \'Asia/Kolkata\', \'start_date\': start.isoformat(), \'days\': days,\n            \'rooms\': 1, \'adults\': 1, \'children\': 0, \'currency\': \'INR\', \'stay_nights\': 1}\n\n\ndef price_value(label):\n    """Never turn Google\'s abbreviated K labels into exact amounts."""\n    match = PRICE.fullmatch(label.strip()) if isinstance(label, str) else None\n    if not match:\n        return None\n    try:\n        value = Decimal(match[1].replace(\',\', \'\'))\n    except InvalidOperation:\n        return None\n    if not value.is_finite() or value <= 0:\n        return None\n    abbreviated = bool(match[2])\n    return {\'amount\': None if abbreviated else float(value),\n            \'approximate_amount\': float(value * 1000) if abbreviated else None,\n            \'display_amount\': label.strip(),\n            \'precision\': \'abbreviated\' if abbreviated else \'displayed_integer\'}\n\n\ndef one_traveler(snapshot):\n    controls = snapshot.get(\'controls\', [])\n    travelers = [c.get(\'aria_label\') for c in controls\n                 if str(c.get(\'aria_label\') or \'\').startswith(\'Number of travelers.\')]\n    return travelers == [TRAVELERS]\n\n\ndef source_identity(snapshot):\n    return (snapshot.get(\'title\') == \'Hotel Aketa - Google hotels\' and snapshot.get(\'url\') == SOURCE_URL\n            and any(re.sub(r\'[\\s\\u200b]\', \'\', c.get(\'text\', \'\')) == \'CurrencyINR\'\n                    for c in snapshot.get(\'controls\', [])))\n\n\ndef _base_row(context, checkin, observed_at, kind, source_path):\n    return {\'hotel_name\': HOTEL_NAME, \'provider_hotel_id\': ENTITY, \'source\': \'google_hotels\',\n            \'checkin\': checkin.isoformat(), \'checkout\': (checkin + timedelta(days=1)).isoformat(),\n            \'rooms\': None, \'requested_rooms\': 1, \'adults\': 1, \'children\': 0, \'currency\': \'INR\',\n            \'room_count_verified\': False, \'party_basis\': \'one adult selected; Google offer may omit room count\',\n            \'room_id\': None, \'room_name\': None, \'rate_plan_id\': None, \'rate_plan_name\': None,\n            \'amount_type\': kind, \'taxes_and_fees\': None, \'taxes_included\': None,\n            \'fees_included\': None, \'meals\': None, \'cancellation\': None,\n            \'availability\': \'price_displayed\', \'direct_supplier_quote\': False,\n            \'provenance\': {\'source_url\': SOURCE_URL, \'observed_at\': observed_at,\n                           \'source_path\': source_path, \'method\': \'rendered_public_google_hotels\'}}\n\n\ndef calendar_rows(snapshot, context, observed_at):\n    """Parse the displayed month/date/price sequence with an explicit year anchor."""\n    if not source_identity(snapshot) or not one_traveler(snapshot):\n        return []\n    text = snapshot.get(\'visible_text\', \'\')\n    if \'Best prices for 1-night stay\' not in text or \'\\nNightly total\\n\' not in text:\n        return []\n    controls = snapshot.get(\'controls\', [])\n    if not any(c.get(\'aria_label\') == \'Price displayedNightly total\' for c in controls):\n        return []\n    lines = text.splitlines()\n    headers = [(i, MONTHS.index(m[1]) + 1, int(m[2]) if m[2] else None)\n               for i, line in enumerate(lines) if (m := MONTH_PATTERN.fullmatch(line.strip()))]\n    anchor = next((j for j, (_, _, year) in enumerate(headers) if year is not None), None)\n    if anchor is None:\n        return []\n    years = {anchor: headers[anchor][2]}\n    for j in range(anchor - 1, -1, -1):\n        years[j] = years[j + 1] - (headers[j][1] > headers[j + 1][1])\n    for j in range(anchor + 1, len(headers)):\n        years[j] = headers[j][2] or years[j - 1] + (headers[j][1] < headers[j - 1][1])\n    start = date.fromisoformat(context[\'start_date\'])\n    end = start + timedelta(days=context[\'days\'])\n    rows = {}\n    conflicts = set()\n    for j, (index, month, explicit_year) in enumerate(headers):\n        year = years[j]\n        if explicit_year is not None and explicit_year != year:\n            return []\n        stop = headers[j + 1][0] if j + 1 < len(headers) else len(lines)\n        for n in range(index + 1, stop - 1):\n            if not re.fullmatch(r\'[0-9]{1,2}\', lines[n].strip()):\n                continue\n            price = price_value(lines[n + 1])\n            if price is None:\n                continue\n            try:\n                day = date(year, month, int(lines[n]))\n            except ValueError:\n                continue\n            if not start <= day < end:\n                continue\n            row = _base_row(context, day, observed_at, \'google_calendar_minimum\',\n                            f"snapshots.{snapshot.get(\'stage\')}.visible_text.lines[{n}:{n+2}]")\n            row.update(price, provider=None, calendar_caption=\'Best prices for 1-night stay\')\n            if day in rows and rows[day][\'display_amount\'] != row[\'display_amount\']:\n                conflicts.add(day)\n            else:\n                rows[day] = row\n    return [rows[day] for day in sorted(rows) if day not in conflicts]\n\n\ndef selected_dates(snapshot, context):\n    start = date.fromisoformat(context[\'start_date\'])\n    result = []\n    for label in (\'Check-in\', \'Check-out\'):\n        inputs = [e.get(\'value\') for e in snapshot.get(\'controls\', []) if e.get(\'aria_label\') == label]\n        if len(inputs) != 1:\n            return None\n        value = inputs[0]\n        found = []\n        for offset in range(context[\'days\'] + 1):\n            day = start + timedelta(days=offset)\n            if value == day.strftime(\'%a, %b \') + str(day.day):\n                found.append(day)\n        if len(found) != 1:\n            return None\n        result.append(found[0])\n    return result if result[1] - result[0] == timedelta(days=1) else None\n\n\ndef partner_rows(snapshot, context, observed_at):\n    if not source_identity(snapshot) or not one_traveler(snapshot):\n        return []\n    dates = selected_dates(snapshot, context)\n    if dates is None or \'\\nNightly total\\n\' not in snapshot.get(\'visible_text\', \'\'):\n        return []\n    text = snapshot[\'visible_text\']\n    marker = \'Sponsored·Featured options\' if \'Sponsored·Featured options\' in text else \'All options\'\n    if marker not in text:\n        return []\n    lines = [x.strip() for x in text.split(marker, 1)[1].splitlines() if x.strip() not in {\'\', \',\'}]\n    provider = None\n    start = 0\n    rows, seen = [], set()\n    for index, line in enumerate(lines):\n        if line in PARTNERS:\n            provider, start = line, index + 1\n            continue\n        price = price_value(line)\n        if index == start and not price and line not in {\'Visit site\', \'All options\'}:\n            description = (re.search(r\'\\bRoom\\b\', line) or re.match(\n                r\'^(?:Free\\b|Official Site$|[0-9]+ guests?\\b|Breakfast\\b|Non-refundable\\b)\', line))\n            if not description:\n                provider = None  # A newly encountered supplier must not inherit the previous one.\n        if not price or provider is None:\n            if line in {\'Visit site\', \'All options\'}:\n                start = index + 1\n            continue\n        descriptions = lines[start:index]\n        combined = \' · \'.join(descriptions)\n        guest_counts = {int(m) for m in re.findall(r\'\\b([0-9]+) guests?\\b\', combined)}\n        if guest_counts != {1}:\n            start = index + 1\n            continue\n        room = next((x for x in descriptions if re.search(r\'\\bRoom\\b\', x)\n                     and not re.search(r\'guest|bed|breakfast\', x, re.I)), None)\n        key = (provider, room, price[\'display_amount\'], combined)\n        if key in seen:\n            continue\n        seen.add(key)\n        row = _base_row(context, dates[0], observed_at, \'google_partner_nightly_total\',\n                        f"snapshots.{snapshot.get(\'stage\')}.visible_text.partner_lines[{start}:{index+1}]")\n        row.update(price, provider=provider, room_name=room, source_description=combined,\n                   meals=\'breakfast\' if (any(part.strip().lower() in {\'breakfast\', \'free breakfast\'}\n                                             for part in combined.split(\'·\'))\n                                           and not re.search(r\'breakfast not included|no breakfast|without breakfast|excluding breakfast\', combined, re.I)) else None,\n                   cancellation=next((x for x in descriptions if \'cancellation\' in x.lower()), None))\n        rows.append(row)\n        start = index + 1\n    return rows\n\n\ndef build_report(capture, context=None):\n    context = requested_context() if context is None else context\n    expected = requested_context(context.get(\'start_date\'), context.get(\'days\'))\n    if any(context.get(key) != expected[key] or type(context.get(key)) is not type(expected[key])\n           for key in expected):\n        raise ValueError(\'Hotel context must be one adult, zero children, INR, one night, 30 dates\')\n    observed_at = capture.get(\'observed_at\')\n    observed_time = datetime.fromisoformat(observed_at)\n    if observed_time.tzinfo is None:\n        raise ValueError(\'Observation time must include its timezone\')\n    snapshots = capture.get(\'snapshots\', [])\n    # Only a source-proven adult decrement with zero children binds the traveler\n    # control to an adult. An arbitrary one-traveler label is insufficient.\n    party_proofs = [i for i, s in enumerate(snapshots) if source_identity(s) and any(c.get(\'aria_label\') == \'Remove adult\' and c.get(\'disabled\') is True\n                         for c in s.get(\'controls\', [])) and\n                      any(c.get(\'aria_label\') == \'Remove child\' and c.get(\'disabled\') is True\n                          for c in s.get(\'controls\', []))]\n    status_good = type(capture.get(\'browser_status\')) is int and capture[\'browser_status\'] == 200\n    calendar, partners = [], []\n    calendar_values, calendar_conflicts = {}, set()\n    if party_proofs and status_good and not capture.get(\'stop_reason\'):\n        for i, s in enumerate(snapshots):\n            if not any(index < i for index in party_proofs):\n                continue\n            parsed = calendar_rows(s, context, observed_at)\n            for row in parsed:\n                day = row[\'checkin\']\n                if day in calendar_values and calendar_values[day][\'display_amount\'] != row[\'display_amount\']:\n                    calendar_conflicts.add(day)\n                else:\n                    calendar_values[day] = row\n            parsed = partner_rows(s, context, observed_at)\n            if parsed:\n                partners = parsed\n    calendar = [calendar_values[day] for day in sorted(calendar_values) if day not in calendar_conflicts]\n    rates = calendar + partners\n    start = date.fromisoformat(context[\'start_date\'])\n    dates = []\n    for i in range(context[\'days\']):\n        day = start + timedelta(days=i)\n        rows = [r for r in rates if r[\'checkin\'] == day.isoformat()]\n        dates.append({\'checkin\': day.isoformat(), \'checkout\': (day + timedelta(days=1)).isoformat(),\n                      \'state\': \'indicative_price\' if rows else \'unknown\',\n                      \'reason\': None if rows else \'source_did_not_return_verified_price\', \'rate_count\': len(rows)})\n    priced = sum(row[\'state\'] == \'indicative_price\' for row in dates)\n    return {\'state\': \'complete_indicative_calendar\' if len(calendar) == context[\'days\'] else \'partial\',\n            \'requested_context\': context,\n            \'context\': {**context, \'rooms\': None, \'requested_rooms\': 1, \'room_count_verified\': False},\n            \'observed_at\': observed_at, \'dates\': dates, \'rates\': rates,\n            \'summary\': {\'date_count\': context[\'days\'], \'indicative_dates\': priced, \'calendar_price_rows\': len(calendar),\n                        \'partner_offer_dates\': len({r[\'checkin\'] for r in partners}),\n                        \'partner_offer_rows\': len(partners), \'quoted_dates\': 0, \'unavailable_dates\': 0,\n                        \'unknown_dates\': context[\'days\'] - priced, \'rate_rows\': len(rates),\n                        \'abbreviated_calendar_rows\': sum(r[\'precision\'] == \'abbreviated\' for r in calendar)},\n            \'stop_reason\': capture.get(\'stop_reason\'),\n            \'limitations\': [\'Google-displayed indicative prices; direct supplier checkout quotes are not verified.\',\n                           \'Room count, tax inclusion and fee inclusion are not explicitly verified by this page.\',\n                           \'Calendar minimums omit supplier, room, meal and cancellation details.\',\n                           \'Abbreviated K prices have no exact amount; unknown cells never mean unavailable.\']}\n\n\ndef export_capture(source: Path, output: Path = DEFAULT_OUTPUT, *, start_date=None):\n    capture = json.loads(source.read_text(encoding=\'utf-8-sig\'))\n    report = build_report(capture, requested_context(start_date))\n    report[\'source_artifact\'] = str(source.resolve())\n    output.mkdir(parents=True, exist_ok=True)\n    (output / \'latest.json\').write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding=\'utf-8\')\n    fields = [\'checkin\', \'checkout\', \'provider\', \'room_name\', \'amount_type\', \'amount\', \'approximate_amount\',\n              \'display_amount\', \'precision\', \'currency\', \'requested_rooms\', \'rooms\', \'adults\', \'children\',\n              \'meals\', \'cancellation\', \'taxes_and_fees\', \'taxes_included\', \'fees_included\', \'source_description\',\n              \'direct_supplier_quote\', \'room_count_verified\', \'source_url\', \'observed_at\', \'source_path\']\n    with (output / \'rates.csv\').open(\'w\', newline=\'\', encoding=\'utf-8-sig\') as handle:\n        writer = csv.DictWriter(handle, fieldnames=fields, extrasaction=\'ignore\')\n        writer.writeheader()\n        writer.writerows({**row, **row[\'provenance\']} for row in report[\'rates\'])\n    with (output / \'calendar.csv\').open(\'w\', newline=\'\', encoding=\'utf-8-sig\') as handle:\n        writer = csv.DictWriter(handle, fieldnames=[\'checkin\', \'checkout\', \'state\', \'reason\', \'rate_count\'])\n        writer.writeheader()\n        writer.writerows(report[\'dates\'])\n    return report\n\n\n# Actual selectors and labels observed in the September 28 canary. The current\n# DOM must still expose each control uniquely before it is used.\nGOOGLE_SNAPSHOT_JS = r"""() => {const vis=e=>!!e.getClientRects().length && getComputedStyle(e).visibility!==\'hidden\';const selector=e=>{if(e.id)return \'#\'+CSS.escape(e.id);const ps=[];while(e&&e!==document.body){const ss=Array.from(e.parentElement?.children||[]).filter(n=>n.tagName===e.tagName);ps.unshift(e.tagName.toLowerCase()+\':nth-of-type(\'+(ss.indexOf(e)+1)+\')\');e=e.parentElement;}return \'body > \'+ps.join(\' > \');};const raw=document.body.innerText||\'\';return {title:document.title,visible_text:raw.split(\'8 top things to know\')[0].slice(0,8000),controls:Array.from(document.querySelectorAll(\'button,input,[role="button"],[role="combobox"],[role="gridcell"],[role="spinbutton"]\')).filter(vis).slice(0,900).map(e=>({selector:selector(e),tag:e.tagName,id:e.id,text:(e.innerText||\'\').slice(0,180),aria_label:e.getAttribute(\'aria-label\'),aria_disabled:e.getAttribute(\'aria-disabled\'),disabled:!!e.disabled,role:e.getAttribute(\'role\'),value:e.value||\'\',type:e.type||\'\'}))}}"""\n\ndef capture_google(*, output: Path = DEFAULT_OUTPUT):\n    """One browser/session: select one adult, read the displayed 1-night calendar.\n\n    No endpoint replay, proxy changes or supplier navigation. Four bounded form\n    actions expose an entire calendar; source drift ends the read with unknowns.\n    """\n    from scrapling.fetchers import DynamicSession\n    from urllib.parse import urlsplit, urlunsplit\n    from .collect import _quiet_scrapling\n\n    capture = {\'source\': \'google_hotels\', \'source_url\': SOURCE_URL,\n               \'observed_at\': datetime.now(timezone.utc).isoformat(),\n               \'stop_reason\': None, \'snapshots\': [], \'actions\': [],\n               \'browser_navigations\': 1, \'direct_replays\': 0}\n\n    def snapshot(page, stage):\n        item = page.evaluate(GOOGLE_SNAPSHOT_JS)\n        parsed = urlsplit(page.url)\n        item.update(stage=stage, url=urlunsplit((parsed.scheme, parsed.hostname or \'\', parsed.path, \'\', \'\')))\n        capture[\'snapshots\'].append(item)\n        if re.search(r\'captcha|unusual traffic|verify (?:that )?you(?: are|.re) human|access denied\',\n                     item.get(\'visible_text\', \'\')[:2000], re.I):\n            capture[\'stop_reason\'] = \'challenge_detected\'\n        if not source_identity(item):\n            capture[\'stop_reason\'] = capture[\'stop_reason\'] or \'hotel_or_currency_not_verified\'\n        return item\n\n    def setup(page):\n        def observed(response):\n            parsed = urlsplit(response.url)\n            if parsed.hostname == \'www.google.com\' and response.request.resource_type in {\'document\', \'xhr\', \'fetch\'}:\n                if response.status in {401, 403, 429}:\n                    capture[\'stop_reason\'] = f\'access_or_rate_limit_{response.status}\'\n        page.on(\'response\', observed)\n\n    def action(page):\n        try:\n            page.wait_for_timeout(5000)\n            current = snapshot(page, \'initial\')\n            for step in range(4):\n                if capture[\'stop_reason\']:\n                    return\n                controls = current.get(\'controls\', [])\n                if step == 0:\n                    choices = [c for c in controls if c.get(\'aria_label\') ==\n                               \'Number of travelers. Current number of travelers is 2.\']\n                elif step == 1:\n                    choices = [c for c in controls if c.get(\'aria_label\') == \'Remove adult\']\n                elif step == 2:\n                    choices = [c for c in controls if c.get(\'text\') == \'Done\']\n                else:\n                    choices = [c for c in controls if c.get(\'aria_label\') == \'Check-in\']\n                if len(choices) != 1:\n                    capture[\'stop_reason\'] = f\'observed_control_not_unique_{step}\'\n                    return\n                selected = choices[0]\n                control = page.locator(selected[\'selector\'])\n                if control.count() != 1 or not control.is_visible() or not control.is_enabled():\n                    capture[\'stop_reason\'] = \'observed_control_not_actionable\'\n                    return\n                control.click(timeout=5000)\n                capture[\'actions\'].append({\'action\': \'click\', \'label\': selected.get(\'aria_label\') or selected.get(\'text\')})\n                page.wait_for_timeout(2500)\n                current = snapshot(page, f\'action {step+1}\')\n        except Exception as exc:\n            capture[\'stop_reason\'] = \'browser_action_error\'\n            capture[\'error_type\'] = type(exc).__name__\n\n    with _quiet_scrapling():\n        try:\n            with DynamicSession(headless=True, executable_path=r\'C:\\Program Files\\Google\\Chrome\\Application\\chrome.exe\',\n                                retries=1, timeout=45000, google_search=False,\n                                locale=\'en-IN\', timezone_id=\'Asia/Kolkata\') as browser:\n                response = browser.fetch(SOURCE_URL, page_setup=setup, page_action=action, network_idle=False, wait=500)\n                capture[\'browser_status\'] = response.status\n                if response.status in {401, 403, 429}:\n                    capture[\'stop_reason\'] = f\'access_or_rate_limit_{response.status}\'\n        except Exception as exc:\n            capture[\'stop_reason\'] = \'browser_error\'\n            capture[\'error_type\'] = type(exc).__name__\n    output.mkdir(parents=True, exist_ok=True)\n    name = \'google-calendar-\' + datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S%fZ\') + \'.json\'\n    source = output / name\n    source.write_text(json.dumps(capture, indent=2, ensure_ascii=False), encoding=\'utf-8\')\n    return source\n\n\nif __name__ == \'__main__\':\n    import argparse\n    parser = argparse.ArgumentParser(description=__doc__)\n    mode = parser.add_mutually_exclusive_group(required=True)\n    mode.add_argument(\'--source\', type=Path, help=\'Saved observed Google hotel form capture\')\n    mode.add_argument(\'--live\', action=\'store_true\', help=\'One bounded browser calendar read using observed controls\')\n    parser.add_argument(\'--start-date\', required=True, help=\'Frozen local window start YYYY-MM-DD\')\n    args = parser.parse_args()\n    if args.live and args.start_date != requested_context()[\'start_date\']:\n        parser.error(\'A live capture must freeze the current hotel-local date\')\n    source = capture_google() if args.live else args.source\n    result = export_capture(source, start_date=args.start_date)\n    print(json.dumps({\'state\': result[\'state\'], \'summary\': result[\'summary\']}, indent=2))\n', 'compset/google_calendar_network.py': '"""One public Aketa calendar capture with sanitized request-contract discovery."""\nfrom __future__ import annotations\n\nimport argparse\nfrom datetime import date, datetime, timezone\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport re\nimport time\nfrom urllib.parse import parse_qs, urlsplit, urlunsplit\n\nfrom .hotel_aketa import (SOURCE_URL, GOOGLE_SNAPSHOT_JS, build_report,\n                          export_capture, price_value, requested_context, source_identity)\n\n\ndef decode_rpc(text):\n    """Read XSSI-prefixed, optionally length-framed JSON without executing it."""\n    if text.startswith(")]}\'"):\n        text = text.split(\'\\n\', 1)[-1]\n    decoder, cursor, values = json.JSONDecoder(), 0, []\n    while cursor < len(text):\n        while cursor < len(text) and text[cursor].isspace():\n            cursor += 1\n        if cursor == len(text):\n            break\n        try:\n            value, end = decoder.raw_decode(text, cursor)\n        except ValueError:\n            break\n        cursor = end\n        if isinstance(value, list):\n            values.extend(value)\n    decoded = []\n    for row in values:\n        if isinstance(row, list) and len(row) > 2 and row[0] == \'wrb.fr\':\n            try:\n                payload = json.loads(row[2]) if isinstance(row[2], str) else row[2]\n            except ValueError:\n                payload = None\n            decoded.append({\'rpc_id\': row[1], \'payload\': payload})\n    return decoded\n\n\ndef shape(value, depth=0):\n    if depth >= 5:\n        return {\'kind\': type(value).__name__, \'length\': len(value) if isinstance(value, (list, dict, str)) else None}\n    if isinstance(value, list):\n        return [shape(x, depth + 1) for x in value[:8]] + ([{\'omitted\': len(value)-8}] if len(value) > 8 else [])\n    if isinstance(value, dict):\n        return {str(k): shape(v, depth + 1) for k, v in list(value.items())[:12]}\n    return type(value).__name__\n\n\ndef safe_leaf(value, depth=0):\n    if depth > 6:\n        return \'<depth>\'\n    if isinstance(value, list):\n        return [safe_leaf(v, depth+1) for v in value[:20]]\n    if isinstance(value, dict):\n        return {k: safe_leaf(v, depth+1) for k, v in list(value.items())[:20]}\n    if isinstance(value, str):\n        if re.fullmatch(r\'\\d{4}-\\d{2}-\\d{2}|INR|USD|AED|₹[\\d,.K]+\', value):\n            return value\n        return f\'<string:{len(value)}>\'\n    return value\n\n\ndef date_structures(value):\n    found = []\n    def walk(node, path, parent=None, parent_path=None):\n        if len(found) >= 12:\n            return\n        is_date = isinstance(node, list) and len(node) == 3 and all(type(v) is int for v in node)\n        if is_date and 2025 <= node[0] <= 2030 and 1 <= node[1] <= 12 and 1 <= node[2] <= 31:\n            found.append({\'date_path\': path, \'date\': node, \'parent_path\': parent_path,\n                          \'parent\': safe_leaf(parent)})\n            return\n        if isinstance(node, str) and re.fullmatch(r\'202[5-9]-\\d{2}-\\d{2}\', node):\n            found.append({\'date_path\': path, \'date\': node, \'parent_path\': parent_path,\n                          \'parent\': safe_leaf(parent)})\n        elif isinstance(node, list):\n            for i, child in enumerate(node):\n                walk(child, f\'{path}[{i}]\', node, path)\n        elif isinstance(node, dict):\n            for k, child in node.items():\n                walk(child, f\'{path}.{k}\', node, path)\n    walk(value, \'$\')\n    return found\n\n\ndef project_calendar(payload):\n    """Project only the observed dated price records; never infer unavailability."""\n    if not isinstance(payload, list) or len(payload) != 2 or not isinstance(payload[1], list):\n        return []\n    rows = []\n    for item in payload[1]:\n        if not isinstance(item, list) or len(item) <= 15 or item[15] != \'INR\':\n            continue\n        money, stay = item[1], item[8]\n        if not isinstance(money, list) or len(money) < 5:\n            continue\n        display = price_value(money[0])\n        if not display or display.get(\'amount\') is None or type(money[4]) is not int \\\n                or money[4] <= 0 or display[\'amount\'] != money[4]:\n            continue\n        if not isinstance(stay, list) or len(stay) < 2:\n            continue\n        try:\n            if not all(isinstance(d, list) and len(d) == 3 and all(type(n) is int for n in d)\n                       for d in stay[:2]):\n                continue\n            checkin, checkout = date(*stay[0]), date(*stay[1])\n            if (checkout-checkin).days != 1:\n                continue\n        except (ValueError, TypeError):\n            continue\n        rows.append({\'checkin\': checkin.isoformat(), \'checkout\': checkout.isoformat(),\n                     \'currency\': \'INR\', \'observed_money_fields\': safe_leaf(money),\n                     \'amount\': money[4], \'display_amount\': money[0],\n                     \'precision\': \'displayed_integer\', \'amount_type\': \'google_calendar_minimum\',\n                     \'taxes_included\': None, \'fees_included\': None,\n                     \'direct_supplier_quote\': False, \'room_count_verified\': False})\n    unique, conflicts = {}, set()\n    for row in rows:\n        day = row[\'checkin\']\n        if day in unique and unique[day][\'amount\'] != row[\'amount\']:\n            conflicts.add(day)\n        unique[day] = row\n    return [row for day, row in sorted(unique.items()) if day not in conflicts]\n\n\ndef compare_calendar(rows, report):\n    displayed = {r[\'checkin\']: r[\'amount\'] for r in report[\'rates\']\n                 if r[\'amount_type\'] == \'google_calendar_minimum\' and r.get(\'amount\') is not None}\n    by_date = {r[\'checkin\']: r for r in rows}\n    matches = []\n    for field in (4, 2):\n        for rounding, convert in [(\'exact\', lambda x:x), (\'ceil\', math.ceil), (\'round\', round)]:\n            count = 0\n            for day, price in displayed.items():\n                money = by_date.get(day, {}).get(\'observed_money_fields\', [])\n                if len(money) > field and type(money[field]) in {int, float} and math.isfinite(money[field]):\n                    count += convert(money[field]) == price\n            matches.append({\'money_index\': field, \'display_rounding\': rounding, \'matched_dates\': count})\n    exact_matches = sum(type(by_date.get(day, {}).get(\'amount\')) is int\n                        and by_date[day][\'amount\'] == price\n                        for day, price in displayed.items())\n    return {\'matched_dates\': exact_matches, \'displayed_dates\': len(displayed),\n            \'all_displayed_dates_match\': bool(displayed) and exact_matches == len(displayed),\n            \'rounding_diagnostics\': matches}\n\n\ndef verified_replay_frame(replay, selected):\n    if replay.get(\'status\') != 200 or selected.get(\'request_contains_known_hotel_entity\') is not True:\n        return None\n    matched = [frame for frame in replay.get(\'frames\', [])\n               if frame.get(\'display_comparison\', {}).get(\'all_displayed_dates_match\') is True]\n    return matched[0] if len(matched) == 1 else None\n\n\ndef observation_for_index(observations, index):\n    matches = [item for item in observations\n               if type(item.get(\'index\')) is int and item[\'index\'] == index]\n    return matches[0] if len(matches) == 1 else {}\n\n\ndef write_replay(path, replay):\n    status = replay.get(\'status\')\n    if status in {401, 403, 429}:\n        replay[\'stop_reason\'] = f\'access_or_rate_limit_{status}\'\n    Path(path).write_text(json.dumps(replay, indent=2), encoding=\'utf-8\')\n\n\ndef discover(output, *, hold_seconds=0):\n    from scrapling.fetchers import DynamicSession\n    from .collect import _quiet_scrapling\n\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=False)\n    total_started = time.monotonic()\n    context = requested_context()\n    capture = {\'source\': \'google_hotels\', \'source_url\': SOURCE_URL,\n               \'observed_at\': datetime.now(timezone.utc).isoformat(), \'stop_reason\': None,\n               \'snapshots\': [], \'actions\': [], \'browser_navigations\': 1, \'direct_replays\': 0}\n    observed, private = [], []\n    phase = {\'value\': \'navigation\'}\n    counts = {\'browser_requests\': 0, \'google_requests\': 0, \'rpc_responses\': 0, \'replay_requests\': 0}\n\n    def snapshot(page, stage):\n        item = page.evaluate(GOOGLE_SNAPSHOT_JS)\n        parsed = urlsplit(page.url)\n        item.update(stage=stage, url=urlunsplit((parsed.scheme, parsed.hostname or \'\', parsed.path, \'\', \'\')))\n        capture[\'snapshots\'].append(item)\n        if re.search(r\'captcha|unusual traffic|verify (?:that )?you(?: are|.re) human|access denied\',\n                     item.get(\'visible_text\', \'\')[:2000], re.I):\n            capture[\'stop_reason\'] = \'challenge_detected\'\n        if not source_identity(item):\n            capture[\'stop_reason\'] = capture[\'stop_reason\'] or \'hotel_or_currency_not_verified\'\n        return item\n\n    def setup(page):\n        def requested(request):\n            counts[\'browser_requests\'] += 1\n            if urlsplit(request.url).hostname == \'www.google.com\':\n                counts[\'google_requests\'] += 1\n        def response_ready(response):\n            parsed = urlsplit(response.url)\n            if parsed.hostname != \'www.google.com\':\n                return\n            if response.status in {401, 403, 429}:\n                if phase[\'value\'] != \'replay\':\n                    capture[\'stop_reason\'] = f\'access_or_rate_limit_{response.status}\'\n                return\n            if \'batchexecute\' not in parsed.path:\n                return\n            counts[\'rpc_responses\'] += 1\n            try:\n                text = response.text()\n                request = response.request\n                fields = parse_qs(request.post_data or \'\')\n                f_req = json.loads(fields.get(\'f.req\', [\'null\'])[0])\n                decoded = decode_rpc(text)\n                index = len(private)\n                private.append({\'url\': request.url, \'post_data\': request.post_data,\n                                \'headers\': request.all_headers(), \'frames\': decoded})\n                observed.append({\'index\': index, \'phase\': phase[\'value\'], \'status\': response.status,\n                    \'method\': request.method, \'endpoint\': parsed.scheme+\'://\'+parsed.hostname+parsed.path,\n                    \'request_rpc_ids\': parse_qs(parsed.query).get(\'rpcids\', []),\n                    \'request_contains_known_hotel_entity\': SOURCE_URL.rsplit(\'/\',1)[-1] in (request.post_data or \'\')\n                        or \'/g/12cnx4r7r\' in json.dumps(f_req),\n                    \'request_shape\': shape(f_req), \'body_bytes\': len(text.encode()),\n                    \'body_sha256\': hashlib.sha256(text.encode()).hexdigest(),\n                    \'response_frames\': [{\'rpc_id\': f[\'rpc_id\'], \'shape\': shape(f[\'payload\']),\n                                         \'dated_structures\': date_structures(f[\'payload\'])} for f in decoded]})\n            except Exception as exc:\n                observed.append({\'index\': None, \'phase\': phase[\'value\'], \'status\': response.status,\n                                 \'parse_error\': type(exc).__name__})\n        page.on(\'request\', requested)\n        page.on(\'response\', response_ready)\n\n    def write_discovery():\n        report = build_report(capture, context)\n        for item in observed:\n            if item.get(\'index\') is not None:\n                frames = private[item[\'index\']][\'frames\']\n                for source, saved in zip(frames, item[\'response_frames\']):\n                    rows = project_calendar(source[\'payload\'])\n                    if rows:\n                        saved[\'calendar_rows\'] = rows\n                        saved[\'display_comparison\'] = compare_calendar(rows, report)\n        evidence = {\'observed_at\': capture[\'observed_at\'], \'stop_reason\': capture[\'stop_reason\'],\n            \'summary\': report[\'summary\'], \'requests\': counts, \'rpc_observations\': observed}\n        (output / \'discovery.json\').write_text(json.dumps(evidence, indent=2), encoding=\'utf-8\')\n        return report\n\n    def action(page):\n        try:\n            page.wait_for_timeout(5000)\n            current = snapshot(page, \'initial\')\n            for step in range(4):\n                if capture[\'stop_reason\']:\n                    return\n                controls = current.get(\'controls\', [])\n                if step == 0:\n                    choices = [c for c in controls if c.get(\'aria_label\') ==\n                        \'Number of travelers. Current number of travelers is 2.\']\n                elif step == 1:\n                    choices = [c for c in controls if c.get(\'aria_label\') == \'Remove adult\']\n                elif step == 2:\n                    choices = [c for c in controls if c.get(\'text\') == \'Done\']\n                else:\n                    choices = [c for c in controls if c.get(\'aria_label\') == \'Check-in\']\n                if len(choices) != 1:\n                    capture[\'stop_reason\'] = f\'observed_control_not_unique_{step}\'\n                    return\n                selected = choices[0]\n                control = page.locator(selected[\'selector\'])\n                if control.count() != 1 or not control.is_visible() or not control.is_enabled():\n                    capture[\'stop_reason\'] = \'observed_control_not_actionable\'\n                    return\n                phase[\'value\'] = f\'action_{step+1}\'\n                control.click(timeout=5000)\n                capture[\'actions\'].append({\'action\': \'click\', \'label\': selected.get(\'aria_label\') or selected.get(\'text\')})\n                page.wait_for_timeout(2500)\n                current = snapshot(page, f\'action {step+1}\')\n            capture[\'browser_status\'] = 200\n            capture[\'browser_calendar_seconds\'] = round(time.monotonic()-total_started, 3)\n            report = write_discovery()\n            # Optional network verification must not erase an already successful calendar.\n            (output/\'rendered-before-replay.json\').write_text(json.dumps(report, indent=2), encoding=\'utf-8\')\n            print(json.dumps({\'discovery_ready\': str(output/\'discovery.json\'),\n                              \'summary\': report[\'summary\'], \'rpc_count\': len(observed)}), flush=True)\n            candidates = [(item[\'index\'], n) for item in observed if item.get(\'phase\') == \'action_4\'\n                          and item.get(\'request_contains_known_hotel_entity\') is True\n                          for n, frame in enumerate(item.get(\'response_frames\', []))\n                          if frame.get(\'display_comparison\', {}).get(\'all_displayed_dates_match\')]\n            if len(candidates) == 1:\n                (output/\'replay-command.json\').write_text(json.dumps(\n                    {\'action\':\'replay\', \'index\': candidates[0][0], \'frame\': candidates[0][1]}), encoding=\'utf-8\')\n                hold_seconds_effective = max(hold_seconds, 1)\n            else:\n                hold_seconds_effective = hold_seconds\n            deadline = time.monotonic() + hold_seconds_effective\n            while hold_seconds_effective and not capture[\'stop_reason\'] and time.monotonic() < deadline:\n                command = output / \'replay-command.json\'\n                if command.exists():\n                    value = json.loads(command.read_text(encoding=\'utf-8\'))\n                    if value.get(\'action\') == \'finish\':\n                        break\n                    if value.get(\'action\') != \'replay\' or type(value.get(\'index\')) is not int:\n                        raise ValueError(\'invalid replay command\')\n                    template = private[value[\'index\']]\n                    headers = {k: v for k, v in template[\'headers\'].items()\n                               if not k.startswith(\':\') and not k.lower().startswith(\'sec-\')\n                               and k.lower() not in {\'cookie\', \'host\', \'content-length\', \'connection\',\n                                   \'accept-encoding\', \'origin\', \'referer\', \'user-agent\'}}\n                    # Same browser context, identical observed URL and POST body, once only.\n                    counts[\'replay_requests\'] = 1\n                    replay_started = time.monotonic()\n                    phase[\'value\'] = \'replay\'\n                    try:\n                        result = page.evaluate(\'\'\'async ({url, body, headers}) => {\n                          const controller = new AbortController();\n                          const timer = setTimeout(() => controller.abort(), 20000);\n                          try { const response = await fetch(url, {method:\'POST\', body, headers,\n                            credentials:\'same-origin\', redirect:\'error\', signal:controller.signal});\n                            return {status:response.status, text:await response.text()};\n                          } finally {clearTimeout(timer);}\n                        }\'\'\', {\'url\': template[\'url\'], \'body\': template[\'post_data\'], \'headers\': headers})\n                    except Exception as exc:\n                        message = re.sub(r\'https?://[^\\s]+\', \'<url>\', str(exc).splitlines()[0])[:240]\n                        (output/\'replay.json\').write_text(json.dumps({\'index\':value[\'index\'],\n                            \'status\':None, \'error_type\':type(exc).__name__, \'error\':message,\n                            \'elapsed_seconds\':round(time.monotonic()-replay_started,3)}), encoding=\'utf-8\')\n                        break\n                    frames = decode_rpc(result[\'text\']) if result[\'status\'] == 200 else []\n                    replay = {\'index\': value[\'index\'], \'status\': result[\'status\'],\n                        \'elapsed_seconds\': round(time.monotonic()-replay_started, 3),\n                        \'frames\': [{\'rpc_id\': f[\'rpc_id\'], \'shape\': shape(f[\'payload\']),\n                                    \'dated_structures\': date_structures(f[\'payload\']),\n                                    \'calendar_rows\':project_calendar(f[\'payload\']),\n                                    \'display_comparison\':compare_calendar(project_calendar(f[\'payload\']),report)}\n                                   for f in frames]}\n                    # Preserve only a reviewed date/price subtree, chosen from sanitized discovery.\n                    if \'payload_path\' in value and frames:\n                        path = value[\'payload_path\']\n                        if not isinstance(path, list) or not all(type(x) is int and x >= 0 for x in path):\n                            raise ValueError(\'payload_path must contain array indexes\')\n                        selected_payload = frames[value.get(\'frame\', 0)][\'payload\']\n                        for index in path:\n                            selected_payload = selected_payload[index]\n                        replay[\'selected_payload\'] = safe_leaf(selected_payload)\n                    write_replay(output/\'replay.json\', replay)\n                    selected = observation_for_index(observed, value[\'index\'])\n                    frame = verified_replay_frame(replay, selected)\n                    if frame is not None:\n                        dates = {r[\'checkin\'] for r in report[\'dates\']}\n                        rates = [r for r in frame[\'calendar_rows\'] if r[\'checkin\'] in dates]\n                        api_report = {\'state\':\'complete_indicative_calendar\' if len(rates)==context[\'days\'] else \'partial\',\n                            \'method\':\'observed_same_session_google_calendar_rpc\', \'rpc_id\':frame[\'rpc_id\'],\n                            \'source_url\':SOURCE_URL, \'hotel_name\':\'Hotel Aketa\',\n                            \'observed_at\':datetime.now(timezone.utc).isoformat(),\n                            \'requested_context\':context, \'observed_context\':report[\'context\'],\n                            \'rates\':rates, \'summary\':{\'requested_dates\':context[\'days\'], \'api_price_rows\':len(rates),\n                                \'response_date_rows\':len(frame[\'calendar_rows\']),\n                                \'matching_displayed_dates\':frame[\'display_comparison\'][\'matched_dates\'],\n                                \'unknown_dates\':context[\'days\']-len(rates), \'unavailable_dates\':0},\n                            \'provenance\':{\'request_index\':value[\'index\'], \'same_observed_url_and_body\':True,\n                                \'request_contains_known_hotel_entity\':selected[\'request_contains_known_hotel_entity\'],\n                                \'identity_basis\':\'known hotel page identity, observed one-adult form, identical request, full date-price display comparison\',\n                                \'rpc_fixture\':str(output/\'replay.json\'), \'browser_capture\':str(output/\'calendar-capture.json\')},\n                            \'timings\':{\'browser_calendar_seconds\':capture[\'browser_calendar_seconds\'],\n                                       \'same_session_replay_seconds\':replay[\'elapsed_seconds\']},\n                            \'limitations\':[\'Indicative displayed calendar minimums, not final supplier checkout quotes.\',\n                                \'Room count, taxes, fees, supplier and cancellation inclusions remain unverified.\',\n                                \'Current Aketa one-adult session only; no multi-property batch or long-lived token claim.\']}\n                        (output/\'api-calendar.json\').write_text(json.dumps(api_report,indent=2), encoding=\'utf-8\')\n                    capture[\'direct_replays\'] = 1\n                    print(json.dumps({\'replay_status\': result[\'status\'], \'artifact\': str(output/\'replay.json\')}), flush=True)\n                    break\n                page.wait_for_timeout(500)\n        except Exception as exc:\n            capture[\'stop_reason\'] = capture[\'stop_reason\'] or \'browser_action_error\'\n            capture[\'error_type\'] = type(exc).__name__\n\n    with _quiet_scrapling():\n        try:\n            with DynamicSession(headless=True, executable_path=None,\n                                retries=1, timeout=45000, google_search=False,\n                                locale=\'en-IN\', timezone_id=\'Asia/Kolkata\') as browser:\n                response = browser.fetch(SOURCE_URL, page_setup=setup, page_action=action, network_idle=False, wait=500)\n                capture[\'browser_status\'] = response.status\n        except Exception as exc:\n            capture[\'stop_reason\'] = capture[\'stop_reason\'] or \'browser_error\'\n            capture[\'error_type\'] = type(exc).__name__\n    source = output/\'calendar-capture.json\'\n    source.write_text(json.dumps(capture, indent=2, ensure_ascii=False), encoding=\'utf-8\')\n    report = export_capture(source, output, start_date=context[\'start_date\'])\n    write_discovery()\n    private.clear()\n    return {\'state\': report[\'state\'], \'summary\': report[\'summary\'], \'counts\': counts,\n            \'stop_reason\': capture[\'stop_reason\'], \'total_seconds\':round(time.monotonic()-total_started,3),\n            \'api_calendar_created\':(output/\'api-calendar.json\').exists(), \'output\': str(output)}\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--output\', required=True, type=Path)\n    parser.add_argument(\'--hold-seconds\', type=int, default=0)\n    args = parser.parse_args()\n    if not 0 <= args.hold_seconds <= 240:\n        parser.error(\'--hold-seconds must be 0..240\')\n    print(json.dumps(discover(args.output, hold_seconds=args.hold_seconds), indent=2), flush=True)\n'}
for name, code in modules.items():
    target = PACKAGE_ROOT / name
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        assert target.read_text(encoding="utf-8") == code, "Source drift: stop"
    else:
        with target.open("x", encoding="utf-8") as stream:
            stream.write(code)
    if target.name in source_manifest:
        actual = hashlib.sha256(target.read_bytes()).hexdigest()
        assert actual == source_manifest[target.name]["packaged_sha256"]
print({"verified_modules": len(modules), "source_manifest": source_manifest})


## Steps
### 4. Collect one new calendar
At most 240 seconds. Stop on controls/identity/context drift, a challenge, 401/403/429 or source failure. Do not repeatedly rerun the collection cell.

In [ ]:
import os, subprocess, sys
from datetime import datetime, timezone
from pathlib import Path
import uuid

run_id = 'aketa-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
RUN = Path('/content/compset-runs') / run_id
assert not RUN.exists(), 'Never overwrite a prior run'
# The child process avoids the notebook event-loop conflict of sync Playwright.
# Only one fresh, anonymous direct Google session; no proxy or challenge retries.
command = [sys.executable, '-m', 'compset.google_calendar_network', '--output', str(RUN)]
environment = dict(os.environ, PYTHONPATH=str(PACKAGE_ROOT), PYTHONIOENCODING='utf-8')
try:
    completed = subprocess.run(command, cwd=PACKAGE_ROOT, env=environment,
                               capture_output=True, text=True, timeout=240)
    print({'collector_exit': completed.returncode, 'run_id': run_id})
except subprocess.TimeoutExpired:
    raise RuntimeError('Finite collector budget exhausted; do not retry automatically') from None
assert (RUN / 'latest.json').is_file(), 'No retained collector report: do not retry automatically'
report = json.loads((RUN / 'latest.json').read_text(encoding='utf-8'))
print({'state': report['state'], 'summary': report['summary'], 'stop_reason': report['stop_reason']})
if report.get('stop_reason'):
    print('Source attempt stopped; save diagnostics only. No challenge bypass or fallback.')


## Checks
### 5. Copy allowed outputs and re-read their digests
Each run gets a new folder. Unknown values stay unknown; prior source timestamps are not refreshed by copying. No opaque request bodies/cookies/headers are exported.

In [ ]:
import hashlib, json, shutil

allowed = ('calendar-capture.json', 'discovery.json', 'rendered-before-replay.json',
           'replay.json', 'api-calendar.json', 'latest.json', 'rates.csv', 'calendar.csv')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
destination = DRIVE_ROOT / run_id
destination.mkdir(exist_ok=False)
proof = {'run_id': run_id, 'method': 'Colab standard CPU -> explicitly mounted Drive',
         'source_manifest': source_manifest, 'state': report['state'], 'summary': report['summary'],
         'scope': 'Aketa 30-day indicative canary only', 'files': {},
         'not_acquired': ['365-day coverage', 'peer hotel rates', 'complete OTA quotes',
                         'verified taxes/fees breakup', 'monthly ten-date verification']}
for name in allowed:
    source = RUN / name
    if not source.is_file():
        continue
    content = source.read_bytes()
    assert len(content) <= 2_000_000, 'Unexpectedly large output; stop before export'
    digest = hashlib.sha256(content).hexdigest()
    target = destination / name
    with target.open('xb') as stream:
        stream.write(content)
    assert hashlib.sha256(target.read_bytes()).hexdigest() == digest, 'Drive round-trip mismatch'
    proof['files'][name] = {'bytes': len(content), 'sha256': digest, 'reread_verified': True}
with (destination / 'receipt.json').open('x', encoding='utf-8') as stream:
    json.dump(proof, stream, indent=2)
assert proof['files'], 'No output exported'
print({'drive_folder': str(destination), 'verified_files': len(proof['files']),
       'summary': proof['summary'], 'not_acquired': proof['not_acquired']})
# Original VM files are retained too; no move/delete if Drive is interrupted.


## Next Steps
Colab execution, rendered output inspection and Drive round-trip remain unverified until these cells actually run. Extend only after the canary: verify peer Google identities, inspect normal month navigation, implement bounded serial coverage/checkpoints, and compare actual supplier quotes for taxes/fees. A Google minimum is not the proven lowest price across the entire internet.